# ⚙️ Configuration & Export

How to configure parsing, mask proper nouns, and build multi-project datasets.

## 🎯 Goals of this notebook

1. **Understand RunConfig options** — learn every toggle the parser exposes
2. **Compare outputs** — see how dropping damaged signs or masking POS changes transliterations
3. **Build a multi-project dataset** — parse several corpora and combine into one DataFrame
4. **Export** — save combined data as JSONL and CSV for downstream use
5. **Quick analysis** — explore the combined dataset by project, provenance, and period

## 1. RunConfig options

| Option | Default | What it does |
|---|---|---|
| `limit` | `None` | Only parse first N texts (good for testing) |
| `drop_missing` | `False` | Remove signs marked `[x]` (completely broken) |
| `drop_damaged` | `False` | Remove signs marked `⸢x⸣` (partially damaged) |
| `mask_pos` | `[]` | Replace words of certain POS with `[TAG]` |
| `languages` | `["Akkadian"]` | Which languages to process |
| `use_cache` | `True` | Use cached results if available |

### Discovering Valid Configurations
If you want to know what values are valid for `mask_pos`, `languages`, or `periods`, you can query the bundled reference data directly:

In [ ]:
from oracc_parser.pipeline import reference_data

# See valid POS tags for masking
pos_df = reference_data.get_pos_tags()
print('Valid POS tags (first 15):')
print(pos_df['POS-tag'].head(15).tolist())
print()

# See valid languages
lang_df = reference_data.get_languages()
print('Valid Languages:')
print(lang_df['lang'].tolist())


In [2]:
from oracc_parser import parse_project, RunConfig, get_transliterations

# Compare default vs. cleaned
PROJECT = "saao/saa01"

# Default: keep everything
rec_default = parse_project(PROJECT, config=RunConfig(limit=2))
print("DEFAULT:")
for _, r in get_transliterations(rec_default).iterrows():
    print(f"  {r['id']}: {r['transliteration'][:100]}")

print()

# Cleaned: drop broken signs, mask personal + divine names
rec_clean = parse_project(PROJECT, config=RunConfig(
    limit=2,
    drop_missing=True,
    mask_pos=["PN", "DN"],
))
print("CLEANED (drop broken, mask PN+DN):")
for _, r in get_transliterations(rec_clean).iterrows():
    print(f"  {r['id']}: {r['transliteration'][:100]}")

14:17:46 | INFO    | oracc_parser | Already downloaded: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\data\jsonzip\saao-saa01.zip
Parsing saao/saa01: 100%|██████████| 2/2 [00:00<00:00, 102.73it/s]
14:17:47 | INFO    | oracc_parser | Loaded 2/2 tablets from cache, parsed 0 new
14:17:47 | INFO    | oracc_parser | Already downloaded: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\data\jsonzip\saao-saa01.zip


DEFAULT:
  saao/saa01_P314227: [a-na LUGAL be-li₂]-ia₂
[ARAD-ka {1}ki-ṣir—aš-šur lu] DI-mu a-⸢na⸣
[LUGAL be-li₂-ia₂ ša LUGAL] ⸢be⸣-
  saao/saa01_P313458: [a-na LUGAL] EN-⸢ia⸣ ARAD-⸢ka⸣ {1}hu-un-ni-i]
[lu-u] ⸢šul⸣-mu a-na ⸢LUGAL EN⸣-ia₂ {d}AG\d u {⸢d⸣}[AM



Parsing saao/saa01: 100%|██████████| 2/2 [00:00<00:00, 41.92it/s]
14:17:48 | INFO    | oracc_parser | Loaded 2/2 tablets from cache, parsed 0 new


CLEANED (drop broken, mask PN+DN):
  saao/saa01_P314227: [a-na LUGAL be-li₂]-ia₂
[ARAD-ka PN lu] DI-mu a-⸢na⸣
[LUGAL be-li₂-ia₂ ša LUGAL] ⸢be⸣-li₂ iš-⸢pur-an
  saao/saa01_P313458: [a-na LUGAL] EN-⸢ia⸣ ARAD-⸢ka⸣ PN
[lu-u] ⸢šul⸣-mu a-na ⸢LUGAL EN⸣-ia₂ DN u DN a-na LUGAL EN-ia₂]
[li


## 2. POS masking reference

| Tag | Meaning 
|---|---|
| `PN` | Personal Name
| `DN` | Divine Name
| `GN` | Geographical Name 
| `RN` | Royal Name 
| `TN` | Temple Name

## 3. Build a multi-project dataset

In [3]:
import pandas as pd
from pathlib import Path
from oracc_parser import get_full_flat_table, get_metadata_table
from oracc_parser.settings import data_dir, output_dir

# Find all downloaded projects
zip_dir = data_dir() / "jsonzip"
if not zip_dir.exists():
    zip_dir = Path("../data/jsonzip")

downloaded = sorted([f.stem for f in zip_dir.glob("*.zip")]) if zip_dir.exists() else []
print(f"Found {len(downloaded)} downloaded projects")
print(f"First 10: {downloaded[:10]}")

Found 145 downloaded projects
First 10: ['adsd', 'adsd-adart1', 'adsd-adart2', 'adsd-adart3', 'adsd-adart5', 'adsd-adart6', 'aemw', 'aemw-alalakh-idrimi', 'aemw-amarna', 'aemw-ugarit']


In [4]:
# Parse a few projects and combine
PROJECTS = ["saao/saa01", "saao/saa05"]  # change these to what you want
config = RunConfig(limit=3)  # small limit for demo — remove for full parse

all_dfs = []
for project in PROJECTS:
    print(f"Parsing {project}...")
    try:
        records = parse_project(project, config=config)
        df = get_full_flat_table(records)
        all_dfs.append(df)
        print(f"  → {len(records)} tablets")
    except Exception as e:
        print(f"  → Error: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"\n✓ Combined: {len(combined)} tablets from {len(PROJECTS)} projects")
display(combined)

14:17:48 | INFO    | oracc_parser | Already downloaded: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\data\jsonzip\saao-saa01.zip


Parsing saao/saa01...


Parsing saao/saa01: 100%|██████████| 3/3 [00:00<00:00, 24.70it/s]
14:17:49 | INFO    | oracc_parser | Loaded 3/3 tablets from cache, parsed 0 new
14:17:49 | INFO    | oracc_parser | Already downloaded: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\data\jsonzip\saao-saa05.zip


  → 3 tablets
Parsing saao/saa05...


Parsing saao/saa05: 100%|██████████| 3/3 [00:00<00:00, 190.10it/s]
14:17:50 | INFO    | oracc_parser | Loaded 3/3 tablets from cache, parsed 0 new


  → 3 tablets

✓ Combined: 6 tablets from 2 projects


,id,project,text_id,genre,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,saao/saa01_P314227,saao/saa01,P314227,Administrative Letter,Nineveh,Neo-Assyrian,-911,-612,[a-na LUGAL be-li₂]-ia₂\n[ARAD-ka {1}ki-ṣir—aš...,ana šarri bēlīya\nurdaka Kiṣir-Aššur lū šulmu ...,ana šarru bēlu\nardu Kiṣir-Aššur lū šulmu ana\...,𒀀𒈾 𒈗 𒁁𒉌𒐊\n𒀴𒅗 𒁹𒆠𒈲𒀸𒋩 𒇻 𒁲𒈬 𒀀𒈾\n𒈗 𒁁𒉌𒐊 𒊭 𒈗 𒁁𒉌 𒅖𒁓𒀭𒉌\...,"[To the king], my [lord: your servant Kiṣir-Aš...",107,62
1,saao/saa01_P313458,saao/saa01,P313458,Administrative Letter,Nineveh,Neo-Assyrian,-911,-612,[a-na LUGAL] EN-⸢ia⸣ ARAD-⸢ka⸣ {1}hu-un-ni-i]\...,ana šarri bēlīya urdaka Hunni\nlū šulmu ana ša...,ana šarru bēlu ardu Hunni\nlū šulmu ana šarru ...,𒀀𒈾 𒈗 𒂗𒅀 𒀴𒅗 𒁹𒄷𒌦𒉌𒄿\n𒇻𒌋 𒂄𒈬 𒀀𒈾 𒈗 𒂗𒐊 𒀭𒀝 𒌋 𒀭𒀫𒌓 𒀀𒈾 𒈗 ...,[To the king m]y lord: your servant [Hunnî]. G...,179,167
2,saao/saa01_P334317,saao/saa01,P334317,Administrative Letter,Nineveh,Neo-Assyrian,-911,-612,13 KUŠ₃ mu-lu-u 04 KUŠ₃ DAGAL E₂—dan-ni\n[x KU...,UNK ammāti mūlû UNK ammāti rupšu bēt danni\nUN...,UNK ammatu mūlû UNK ammatu rupšu bītu dannu\nU...,𒌋𒐈 𒌑 𒈬𒇻𒌋 𒐉 𒌑 𒂼 𒂍𒆗𒉌 𒂍𒆗𒉌\nx 𒌑 𒈬𒇻𒌋 𒐉 𒌑 𒂼 𒂍 𒍇𒇻\nx ...,"Height[ 1]3 cubits (6,5 m), width four cubits ...",67,67
3,saao/saa05_P334831,saao/saa05,P334831,Administrative Letter,Nineveh,neo-assyrian,-721,-705,[lu-u DI]-mu a-[na LUGAL EN-ia]\n[ina] ⸢UGU⸣ U...,lū šulmu ana šarri bēlīya\nina muhhi immerē ša...,lū šulmu ana šarru bēlu\nina muhhu immeru ša U...,𒇻𒌋 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀\n𒀸 𒌋𒅗 𒇻𒎌 𒊭 x x 𒊭 𒈗\n𒁁𒉌 𒅖𒁓𒀀𒉌\n𒈠𒀀 ...,"[To the king, my lord: your servant NN. Good h...",119,108
4,saao/saa05_P314281,saao/saa05,P314281,Administrative Letter,Nineveh,neo-assyrian,-717,-706,⸢nu-uk\td 01 me 20 {giš}UR₃⸣-MEŠ\m\nša\ydm LUG...,nuk UNK mē UNK gušūrī\nša šarri alê\nšû issuhu...,nuk UNK meʾatu UNK gušūru\nša šarru ali\nšū sa...,𒉡𒊌 𒁹 𒈨 𒎙 𒄑𒃡𒎌\n𒊭 𒈗 𒀀𒇷𒂊\n𒋗𒌑 𒄿𒋢𒄯 𒅅𒁲𒁉𒀀\n𒈠𒀀 𒄑𒃡𒎌 𒊩𒅆𒌋...,"I (wrote him): ""Where are the king's 120 logs?...",105,74
5,saao/saa05_P334186,saao/saa05,P334186,Administrative Letter,Nineveh,neo-assyrian,-721,-705,a-na LUGAL\p be-li₂-ia\nARAD-ka {1}aš-šur—BAD₃...,ana šarri bēlīya\nurdaka Aššur-dur-paniya\nlū ...,ana šarru bēlu\nardu Aššur-dur-paniya\nlū šulm...,𒀀𒈾 𒈗 𒁁𒉌𒅀\n𒀴𒅗 𒁹𒀸𒋩𒂦𒅆𒅀\n𒇻𒌑 𒂄𒈬 𒀀𒈾 𒈗 𒁁𒉌𒅀\n𒇽𒃲𒐐𒅀 𒇽𒃲𒐐𒅀...,"To the king, my lord: your servant Aššur-dur-p...",165,165


In [5]:
# Export the combined dataset
out = output_dir()
out.mkdir(parents=True, exist_ok=True)

path_jsonl = out / "combined_dataset.jsonl"
path_csv = out / "combined_dataset.csv"

combined.to_json(path_jsonl, orient="records", lines=True, force_ascii=False)
combined.to_csv(path_csv, index=False)

print(f"Saved to:")
print(f"  {path_jsonl}  ({path_jsonl.stat().st_size/1024:.1f} KB)")
print(f"  {path_csv}  ({path_csv.stat().st_size/1024:.1f} KB)")
print(f"\n📁 All output files in {out}:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved to:
  C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\output\combined_dataset.jsonl  (29.0 KB)
  C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\output\combined_dataset.csv  (27.3 KB)

📁 All output files in C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\output:
  combined_dataset.csv                      27.3 KB
  combined_dataset.jsonl                    29.0 KB
  saao_saa01.csv                            26.7 KB
  saao_saa01.jsonl                          28.1 KB


## 4. Quick analysis of the data

In [6]:
print("Texts by project:")
print(combined["project"].value_counts().to_string())

print("\nTexts by provenance:")
print(combined["provenance"].value_counts().to_string())

print("\nTexts by period:")
print(combined["period"].value_counts().to_string())

print(f"\nAvg tokens per text: {combined['total_tokens'].mean():.0f}")
print(f"Total tokens: {combined['total_tokens'].sum()}")

Texts by project:
project
saao/saa01    3
saao/saa05    3

Texts by provenance:
provenance
Nineveh    6

Texts by period:
period
Neo-Assyrian    3
neo-assyrian    3

Avg tokens per text: 124
Total tokens: 742
